### Imports

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

### Paths and parameters

In [22]:
# Dataset paths
RGB_ROOT = r"D:\Datasets\Datasets\EPIC_Kitchen\RGB\P01_02\Original"
ANNOTATION_CSV = r"D:\Datasets\Datasets\EPIC_Kitchen\Label\P01_02.csv"

# Output paths
OUTPUT_FEATURES_CSV = "../Features/EPIC/P01_02_rgb_features.csv"
OUTPUT_LABELS_CSV = "../Labels/EPIC/P01_02_rgb_labeled.csv"

# Model parameters
FEATURE_DIM = 512
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

Using device: cuda


### Load annotation CSV

In [23]:
annotations = pd.read_csv(ANNOTATION_CSV)
print("Total annotations:", len(annotations))
annotations.head()

Total annotations: 145


,StartFrame,EndFrame,Verb,Verb_class,Noun,Noun_class,ActionLabel,ActionName
0,304,410,take,0,plate,4,0,take plate
1,516,556,open,2,bin,25,1,open bin
2,607,1087,throw,8,leftover,209,2,throw leftover
3,1102,1147,close,3,bin,25,3,close bin
4,1260,1353,put-down,1,plate,4,4,put-down plate


### Create frame-level multi-label matrix

In [24]:
max_frame = annotations['EndFrame'].max()
num_classes = annotations['ActionLabel'].max() + 1

print("Total frames:", max_frame)
print("Total classes:", num_classes)

labels = np.zeros((max_frame + 1, num_classes), dtype=np.float32)
for _, row in annotations.iterrows():
    start = int(row['StartFrame'])
    end = int(row['EndFrame'])
    cls = int(row['ActionLabel'])
    labels[start:end+1, cls] = 1.0
print("Labels shape:", labels.shape)

Total frames: 29955
Total classes: 63
Labels shape: (29956, 63)


### Define image transform

In [25]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### Load ResNet50 feature extractor

In [26]:
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
# Remove classifier
modules = list(resnet.children())[:-1]
resnet = nn.Sequential(*modules)
# Projection layer
projection = nn.Linear(2048, FEATURE_DIM)
resnet = resnet.to(DEVICE)
projection = projection.to(DEVICE)
resnet.eval()
projection.eval()

@torch.no_grad()
def extract_feature(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(DEVICE)
    feature = resnet(image)
    feature = feature.view(1, -1)
    feature = projection(feature)
    feature = feature.squeeze(0).cpu().numpy()
    return feature

In [27]:
rgb_frames = sorted(glob.glob(os.path.join(RGB_ROOT, "*.jpg")))
num_frames = len(rgb_frames)
print("Total RGB frames:", num_frames)

Total RGB frames: 30098


### Extract RGB features, Save features and Save Labels

In [28]:
rgb_features = np.zeros((num_frames, FEATURE_DIM), dtype=np.float32)
for i in tqdm(range(num_frames), desc="Extracting RGB features"):
    rgb_features[i] = extract_feature(rgb_frames[i])
print("RGB features shape:", rgb_features.shape)

Extracting RGB features: 100%|███████████████████████████████████████████████████| 30098/30098 [26:57<00:00, 18.61it/s]

RGB features shape: (30098, 512)


In [29]:
### Save RGB features
features_df = pd.DataFrame(rgb_features)
features_df.insert(0, "frame_id", np.arange(num_frames))
features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)
print("Saved RGB features:", OUTPUT_FEATURES_CSV)

Saved RGB features: ../Features/EPIC/P01_02_rgb_features.csv


In [30]:
### Correct alignment fix for labels
label_frames = labels.shape[0]
if label_frames < num_frames:
    padding = np.zeros((num_frames - label_frames, labels.shape[1]), dtype=np.float32)
    labels = np.vstack([labels, padding])
elif label_frames > num_frames:
    labels = labels[:num_frames]
print("Final labels shape:", labels.shape)

Final labels shape: (30098, 63)


In [31]:
### Save labels
labels_df = pd.DataFrame(labels)
labels_df.insert(0, "frame_id", np.arange(num_frames))
labels_df.to_csv(OUTPUT_LABELS_CSV, index=False)
print("Saved labels:", OUTPUT_LABELS_CSV)

Saved labels: ../Labels/EPIC/P01_02_rgb_labeled.csv
